# B2: N-gram Language Model and Perplexity

Build unigram, bigram and trigram models, estimate probabilities with Maximum Likelihood Estimation (MLE), apply smoothing (Laplace), compute perplexity for each model, and compare the models and the effect of smoothing.

**Dataset:** Brown Corpus and Gutenberg Corpus (NLTK). NLTK's `nltk.lm` module provides the n-gram models, smoothing and perplexity.

In [2]:
# nltk : Brown and Gutenberg corpora, and nltk.lm (n-gram language models)
!pip install -q nltk pandas

In [1]:
import random
import pandas as pd
import nltk
for pkg in ["brown", "gutenberg", "punkt", "punkt_tab"]:
    nltk.download(pkg, quiet=True)
from nltk.lm import MLE, Laplace
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_both_ends
from nltk.util import ngrams

## 1. A tiny example first

Three training sentences. `<s>` and `</s>` mark the start and end of a sentence.

In [2]:
train_tiny = [["i", "like", "nlp"], ["i", "like", "python"], ["i", "love", "nlp"]]
print(list(pad_both_ends(train_tiny[0], n=2)))
print(list(ngrams(pad_both_ends(train_tiny[0], n=2), 2)))

['<s>', 'i', 'like', 'nlp', '</s>']
[('<s>', 'i'), ('i', 'like'), ('like', 'nlp'), ('nlp', '</s>')]


In [3]:
# fit a bigram MLE model: P(word | previous word) = count(previous, word) / count(previous)
train_data, vocab = padded_everygram_pipeline(2, train_tiny)
lm = MLE(2)
lm.fit(train_data, vocab)

print("count(i, like):", lm.counts[["i"]]["like"], " count(i):", lm.counts["i"])
print("P(like | i) =", lm.score("like", ["i"]))
print("P(love | i) =", lm.score("love", ["i"]))
print("P(python | love) =", lm.score("python", ["love"]), " <- never seen, so 0")

count(i, like): 2  count(i): 3
P(like | i) = 0.6666666666666666
P(love | i) = 0.3333333333333333
P(python | love) = 0.0  <- never seen, so 0


In [4]:
# Laplace (add-one) smoothing: (count + 1) / (count(previous) + vocabulary size)
train_data, vocab = padded_everygram_pipeline(2, train_tiny)
lap = Laplace(2)
lap.fit(train_data, vocab)
print("vocabulary size:", len(lap.vocab))
print("P(like | i) =", round(lap.score("like", ["i"]), 3))
print("P(python | love) =", round(lap.score("python", ["love"]), 3), " <- no longer 0")

vocabulary size: 8
P(like | i) = 0.273
P(python | love) = 0.111  <- no longer 0


In [5]:
# perplexity of a new sentence: lower is better
test_sent = ["i", "love", "python"]
test_bigrams = list(ngrams(pad_both_ends(test_sent, n=2), 2))
print("MLE perplexity:    ", lm.perplexity(test_bigrams))
print("Laplace perplexity:", round(lap.perplexity(test_bigrams), 2))

MLE perplexity:     inf
Laplace perplexity: 4.97


## 2. Load the corpora

Lowercase words only, then a random 90% / 10% train / test split. We train on 20,000 sentences to keep it fast.

In [6]:
from nltk.corpus import brown, gutenberg

def prepare(sents, n_train=20000, seed=0):
    sents = [[w.lower() for w in s if w.isalpha()] for s in sents]
    sents = [s for s in sents if len(s) > 1]
    random.Random(seed).shuffle(sents)
    cut = int(0.9 * len(sents))
    return sents[:cut][:n_train], sents[cut:][:2000]

corpora = {"Brown": prepare(brown.sents()), "Gutenberg": prepare(gutenberg.sents())}
for name, (tr, te) in corpora.items():
    print(f"{name}: {len(tr)} train sentences, {len(te)} test sentences, e.g. {' '.join(tr[0][:12])}")

Brown: 20000 train sentences, 2000 test sentences, e.g. whatever pole of this contrast one emphasizes and whatever the tension between
Gutenberg: 20000 train sentences, 2000 test sentences, e.g. my sensations were strange


## 3. Unigram, bigram and trigram models with MLE and Laplace, and their perplexity

In [7]:
def train_lm(model_class, n, train):
    data, vocab = padded_everygram_pipeline(n, train)
    model = model_class(n)
    model.fit(data, vocab)
    return model

def perplexity(model, n, test):
    test_ngrams = [g for s in test for g in ngrams(pad_both_ends(s, n=n), n)]
    return model.perplexity(test_ngrams)

rows = []
models = {}
for name, (train, test) in corpora.items():
    for n in [1, 2, 3]:
        for cls in [MLE, Laplace]:
            m = train_lm(cls, n, train)
            models[(name, n, cls.__name__)] = m
            rows.append({"corpus": name, "n": n, "model": cls.__name__, "perplexity": perplexity(m, n, test)})

results = pd.DataFrame(rows).pivot_table(index=["corpus", "model"], columns="n", values="perplexity")
results.columns = ["unigram", "bigram", "trigram"]
results.round(1)

unigram  bigram  trigram
corpus    model                            
Brown     Laplace   1381.1  5330.4  15057.6
          MLE          inf     inf      inf
Gutenberg Laplace    848.3  2656.2  10001.1
          MLE          inf     inf      inf

## 4. The effect of smoothing on single probabilities

In [8]:
bi_mle, bi_lap = models[("Brown", 2, "MLE")], models[("Brown", 2, "Laplace")]
pairs = [("of", "the"), ("the", "same"), ("the", "movie"), ("purple", "elephant")]
pd.DataFrame([{"bigram": f"{a} {b}", "count": bi_mle.counts[[a]][b],
               "MLE": bi_mle.score(b, [a]), "Laplace": bi_lap.score(b, [a])} for a, b in pairs]).round(5)

,bigram,count,MLE,Laplace
0,of the,3579,0.26755,0.09039
1,the same,244,0.00968,0.00476
2,the movie,2,0.00008,0.00006
3,purple elephant,0,0.00000,0.00004


In [9]:
# how much probability Laplace moves away from seen words: P(the | of) before and after
print("count(of):", bi_mle.counts["of"], " vocabulary size:", len(bi_mle.vocab))
print(f"P(the | of): MLE {bi_mle.score('the', ['of']):.3f} -> Laplace {bi_lap.score('the', ['of']):.3f}")

count(of): 13377  vocabulary size: 26228
P(the | of): MLE 0.268 -> Laplace 0.090


## 5. Generate text

In [10]:
for n in [1, 2, 3]:
    m = models[("Brown", n, "MLE")]
    words = m.generate(15, text_seed=["the"] if n > 1 else None, random_seed=4)
    print(f"{n}-gram:", " ".join(w for w in words if w not in ("<s>", "</s>")))

1-gram: door as in but and in until the that design not fill case as day
2-gram: drink had been able to their season the discomfort that he did best could
3-gram: drink for drink until he is the savior of mankind


## Analysis
- **MLE** gives probability 0 to any n-gram never seen in training. One such n-gram in the test set makes the perplexity infinite, which happens for every MLE model here (even the unigram, because test words unseen in training are mapped to `<UNK>`, which has count 0).
- **Laplace** removes the zeros, so every model gets a finite perplexity.
- With Laplace, bigram and trigram perplexities come out **worse** than unigram: adding 1 to every one of the tens of thousands of possible next words takes too much probability away from the words actually seen (P(the | of) drops sharply). More context only helps with better smoothing (Good-Turing, Kneser-Ney, interpolation).
- Generated text shows the effect of context: unigram output is random words, trigram output has locally correct phrases.
- Brown (news, fiction, many genres) and Gutenberg (old books) give different numbers; perplexity is only comparable on the same test set.